# EarthForge — STAC Search & Asset Inspection

This notebook demonstrates the full STAC search workflow:
1. Search a STAC catalog by bounding box and date range
2. Inspect item metadata and available assets
3. Filter items by cloud cover or other properties
4. Fetch (download) selected assets with resume support
5. Validate downloaded COGs

Works with any STAC-compliant API: Element84 Earth Search, Microsoft Planetary Computer, KyFromAbove, USGS 3DEP.

In [ ]:
import sys

sys.path.insert(0, "../../packages/core/src")
sys.path.insert(0, "../../packages/raster/src")
sys.path.insert(0, "../../packages/stac/src")

from earthforge.core.config import EarthForgeProfile

# Earth Search profile (no authentication required)
earth_search = EarthForgeProfile(
    name="earth-search",
    stac_api="https://earth-search.aws.element84.com/v1",
    storage_backend="local",
)

# KyFromAbove profile (Kentucky statewide imagery)
kyfromabove = EarthForgeProfile(
    name="kyfromabove",
    stac_api="https://spved5ihrl.execute-api.us-west-2.amazonaws.com/",
    storage_backend="local",
)

print("Profiles configured")

## 1. Search Sentinel-2 Over Kentucky

Search Element84 Earth Search for Sentinel-2 L2A scenes over Lexington, KY.
The `bbox` parameter is `[west, south, east, north]` in WGS84 (EPSG:4326).

In [ ]:
from earthforge.stac.search import search_catalog

# Lexington, KY bounding box
lexington_bbox = [-84.6, 37.9, -84.4, 38.1]

results = await search_catalog(
    earth_search,
    collections=["sentinel-2-l2a"],
    bbox=lexington_bbox,
    datetime_range="2025-06-01/2025-09-30",
    max_items=10,
)

print(f"Found {len(results.items)} items")
print(f"Search took {results.elapsed_seconds:.3f}s")
print()
for item in results.items:
    print(f"  {item.id}")
    print(f"    Date:   {item.datetime}")
    print(f"    Assets: {item.asset_count}")

## 2. Filter by Cloud Cover

STAC items include `eo:cloud_cover` in their properties.
Filter client-side to keep only clear scenes.

In [ ]:
# Filter to scenes with <20% cloud cover
clear_items = [item for item in results.items if item.properties.get("eo:cloud_cover", 100) < 20]

print(f"Items with <20% cloud cover: {len(clear_items)} of {len(results.items)}")

if clear_items:
    best = min(clear_items, key=lambda i: i.properties.get("eo:cloud_cover", 100))
    print(f"\nClearest scene: {best.id}")
    print(f"  Cloud cover: {best.properties.get('eo:cloud_cover'):.1f}%")
    print(f"  Date: {best.datetime}")

## 3. Inspect Item Assets

Each STAC item carries a dictionary of assets — COG rasters, metadata files, thumbnails.
Inspect which bands are available before downloading.

In [ ]:
if results.items:
    item = results.items[0]
    print(f"Item: {item.id}")
    print(f"Bbox: {item.bbox}")
    print(f"\nAssets ({item.asset_count} total):")
    print(f"  {'Key':<15} {'Media Type':<40} {'Roles'}")
    print(f"  {'-' * 14} {'-' * 39} {'-' * 20}")
    for asset in item.assets:
        roles_str = ", ".join(asset.roles or [])
        print(f"  {asset.key:<15} {(asset.media_type or ''):<40} {roles_str}")

## 4. Read COG Metadata Without Downloading

`inspect_raster` reads only the GeoTIFF header via HTTP range requests.
For a 1 GB Sentinel-2 scene, this transfers ~8 KB, not 1 GB.

In [ ]:
from earthforge.raster.info import inspect_raster

if results.items:
    item = results.items[0]
    # Find the red band (B04 in Sentinel-2)
    red_asset = next((a for a in item.assets if a.key in ("B04", "red")), None)

    if red_asset:
        print(f"Inspecting {red_asset.key} band: {red_asset.href}")
        info = await inspect_raster(red_asset.href)
        print(f"  Dimensions:  {info.width} x {info.height}")
        print(f"  CRS:         {info.crs}")
        print(f"  Bands:       {info.band_count}")
        print(f"  Tiled:       {info.is_tiled} ({info.tile_width}x{info.tile_height})")
        print(f"  Overviews:   {info.overview_count}")
        print(f"  Compression: {info.compression}")

## 5. Fetch Assets

Download specific bands using `fetch_assets`. Downloads run in parallel (default: 4 concurrent).
If the file already exists with the correct byte count, it is skipped (resume support).

**Note:** Sentinel-2 bands are ~100–600 MB each. The cell below fetches only the thumbnail to avoid a large download.

In [ ]:
from earthforge.stac.fetch import fetch_assets

if results.items:
    item = results.items[0]
    # Build the item URL from the search result
    item_url = item.links.get("self") or item.href

    if item_url:
        # Fetch only the thumbnail (small, good for demo)
        fetch_result = await fetch_assets(
            earth_search,
            item_url=item_url,
            assets=["thumbnail"],
            output_dir="./data/sentinel2",
            parallel=2,
        )

        print(f"Item:      {fetch_result.item_id}")
        print(f"Fetched:   {fetch_result.assets_fetched} assets")
        print(f"Skipped:   {fetch_result.assets_skipped} (already complete)")
        print(f"Downloaded:{fetch_result.total_bytes_downloaded:,} bytes")
        print(f"Elapsed:   {fetch_result.elapsed_seconds:.2f}s")
        print()
        for f in fetch_result.files:
            status = "SKIP" if f.skipped else "FETCHED"
            print(f"  [{status}] {f.key} → {f.local_path} ({f.size_bytes:,} bytes)")
    else:
        print("No self link found — run stac search with --include-links to get item URLs")

## 6. KyFromAbove — High-Resolution Orthoimagery

KyFromAbove provides 3-inch orthoimagery and 2-ft DEMs for all of Kentucky.
Phase 3 coverage (2022–present) is fully public with no authentication required.

In [ ]:
# Search KyFromAbove for orthoimagery over Lexington
ky_results = await search_catalog(
    kyfromabove,
    collections=["orthos-phase3"],
    bbox=lexington_bbox,
    max_items=5,
)

print(f"KyFromAbove orthos: {len(ky_results.items)} items")
for item in ky_results.items:
    print(f"  {item.id} | {item.asset_count} assets")

## 7. Serialize Results as JSON

All EarthForge result objects are Pydantic models.
Use `.model_dump_json()` to get clean JSON for downstream processing.

In [ ]:
import json

if results.items:
    # Serialize the first item's summary
    item = results.items[0]
    summary = {
        "id": item.id,
        "datetime": str(item.datetime),
        "bbox": item.bbox,
        "cloud_cover": item.properties.get("eo:cloud_cover"),
        "assets": [a.key for a in item.assets],
    }
    print(json.dumps(summary, indent=2))